# Identifying failure sets

In this notebook we identify maritime edges, as collected in edges_maritime_corrected.gpkg, which overlap with maritime chokepoints.
In case of disruption of a maritime chokepoint these edges would not be avaible to be used by vessels. Thus, a disruption scenario of a particular maritime chokepoint is implemented by removing the maritime edges that interesect it. Here, we provide the code to identify those edges. The list of maritime chokepoints that can be disrupted can be found in the reference behind https://www.nature.com/articles/s41467-025-65403-w.

The output will be collected in failure_set.csv

# Import libraries

In [1]:
import sys
import json
import os

from Chokepoint_Edges import import_data, chokepoint_edge ### Import functions from chokepoint_edges.py



# Data Manipulation
import pandas as pd
import numpy as np


import geopandas as gpd
from geopandas import GeoDataFrame

import dask.dataframe as dd

##### Network libraries #######
import networkx as nx


from pyproj import Geod #### This is to compute the right distance without any bias at the poles.
from itertools import combinations, product ### for iterations of pairs


from shapely.geometry import shape, Point, LineString, Polygon
from shapely.ops import nearest_points
from cartopy import crs as ccrs
from tqdm import tqdm  # ✅ Import tqdm



robinson = ccrs.Robinson(central_longitude=0, globe=None)


# Identify edges that intersect chokekpoints

In [3]:
#### Running the code:

edges_maritime_gdf, chokepoints_df = import_data()

## Use cp_index=None to create the edge mapping for all chokepoints.
chokepoint_edge(
    chokepoints_df,
    edges_maritime_gdf,
    cp_index=None,
    file_name="raw_data/failure_set/intermediate_data/chokepoint_edges",
)


Saved 468 chokepoint-edge mappings to raw_data/failure_set/intermediate_data/chokepoint_edges.csv and raw_data/failure_set/intermediate_data/chokepoint_edges.json


,chokepoint_name,edge_id,from_id,to_id
0,Bab el-Mandeb Strait,maritime130_maritime131,maritime130,maritime131
1,Bab el-Mandeb Strait,maritime130_maritime3312,maritime130,maritime3312
2,Bab el-Mandeb Strait,maritime130_maritime373,maritime130,maritime373
3,Bab el-Mandeb Strait,maritime130_maritime374,maritime130,maritime374
4,Bab el-Mandeb Strait,maritime131_maritime130,maritime131,maritime130
...,...,...,...,...
463,Yucatan Channel,maritime6223_maritime6100,maritime6223,maritime6100
464,Yucatan Channel,maritime6223_port1144,maritime6223,port1144
465,Yucatan Channel,maritime658_maritime490,maritime658,maritime490
466,Yucatan Channel,maritime661_maritime490,maritime661,maritime490


# Pick relevant chokepoints for disruption analysis

In [4]:
edges = pd.read_csv("./raw_data/failure_set/intermediate_data/chokepoint_edges.csv")


In [5]:
chokepoints_df['portname'].unique()

array(['Suez Canal', 'Panama Canal', 'Bosporus Strait',
       'Bab el-Mandeb Strait', 'Malacca Strait', 'Strait of Hormuz',
       'Cape of Good Hope', 'Gibraltar Strait', 'Dover Strait',
       'Oresund Strait', 'Taiwan Strait', 'Korea Strait',
       'Tsugaru Strait', 'Luzon Strait', 'Lombok Strait', 'Ombai Strait',
       'Bohai Strait', 'Torres Strait', 'Sunda Strait', 'Makassar Strait',
       'Magellan Strait', 'Yucatan Channel', 'Windward Passage',
       'Mona Passage', 'Balabac Strait', 'Bering Strait',
       'Mindoro Strait', 'Kerch Strait'], dtype=object)

The code allows for the possibility to select multiple chokepoints. Just add more names into selected_chokepoints. For a list of chokepoints run the cell above.

In [6]:
selected_chokepoints = [
    "Korea Strait"
]


failure_set = edges.loc[
    edges["chokepoint_name"].isin(selected_chokepoints),
    ["edge_id"],
].drop_duplicates()


failure_set.to_csv("./processed_data/damages/failure_set.csv", index=False)